<a href="https://colab.research.google.com/github/erickvaldezsallagos/ingenieria/blob/main/ProyectoIF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PROYECTO DE INGENIERÍA FINANCIERA**

# ***Tema: Apple vs Samsung***

**Integrantes:**\
VALDEZ SALLAGOS ERICK\
AMAYA HERNÁNDEZ JOSÉ LUIS\
REYES FUENTES ESTEPHANY CAROLINA

**TABLA PAGINADA**

**Código**

In [12]:
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets

# 1. CONFIGURACIÓN Y CARGA DE DATOS
# Evita que Pandas corte los textos o enlaces largos con "..."
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', None)

archivo_maestro = 'catalogo_tecnologia_maestro_5500_refinado.csv'
df_maestro = pd.read_csv(archivo_maestro)

# Seleccionamos exactamente las 8 columnas más relevantes y ejecutivas
columnas_relevantes = [
    'marca',
    'nombre',
    'modelo',
    'color',
    'precio',
    'moneda',
    'categoria',
    'url'
]

# 2. DEFINICIÓN DE LA FUNCIÓN INTERACTIVA
def explorar_catalogo_8_columnas(df, filas_por_pagina=12):
    """
    Genera un control deslizante interactivo (slider) para navegar dinámicamente
    a través de todo el catálogo maestro utilizando únicamente las 8 columnas clave.
    """
    total_registros = len(df)
    total_paginas = (total_registros // filas_por_pagina) + (1 if total_registros % filas_por_pagina > 0 else 0)

    # Crear el control deslizante (IntSlider) para cambiar de página de forma visual
    slider_pagina = widgets.IntSlider(
        value=1,
        min=1,
        max=total_paginas,
        step=1,
        description='Página:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='450px')
    )

    # Contenedor de salida para refrescar la tabla dinámicamente sin saturar la consola
    salida = widgets.Output()

    def actualizar_vista(change):
        """Función que se activa automáticamente al mover el slider de página."""
        with salida:
            clear_output(wait=True)
            pagina_actual = slider_pagina.value
            inicio = (pagina_actual - 1) * filas_por_pagina
            fin = inicio + filas_por_pagina

            print(f"--- Mostrando registros del {inicio} al {min(fin, total_registros)} (Página {pagina_actual} de {total_paginas}) ---")
            display(df.iloc[inicio:fin][columnas_relevantes])

    # Observar los cambios en el slider
    slider_pagina.observe(actualizar_vista, names='value')

    # Renderizar la vista inicial (Página 1)
    with salida:
        print(f"--- Mostrando registros del 0 al {filas_por_pagina} (Página 1 de {total_paginas}) ---")
        display(df.iloc[0:filas_por_pagina][columnas_relevantes])

    # Mostrar juntos el control deslizante y el contenedor de la tabla
    display(widgets.VBox([slider_pagina, salida]))

# 3. EJECUCIÓN DE LA INTERFAZ
# Muestra 12 productos por página para mantener un formato visual limpio y ordenado
explorar_catalogo_8_columnas(df_maestro, filas_por_pagina=12)

**JUSTIFICACIÓN:**\
Se utilizaron librerías especializadas como pandas para la manipulación y filtrado estructurado de datos, ipywidgets para la integración de controles gráficos interactivos de búsqueda y navegación, y IPython.display para el renderizado dinámico de tablas ejecutivas, herramientas que en conjunto nos ayudan a transformar un catálogo masivo de más de 6,000 registros en una interfaz ágil, transparente y de alto rendimiento que facilita la auditoría y consulta de los atributos comerciales clave

**RESULTADOS DE BÚSQUEDA**

**Código**

In [17]:
import pandas as pd
from IPython.display import display, HTML
import ipywidgets as widgets  # ipywidgets crear controles e interfaces gráficas interactivas
from ipywidgets import interact

# 1. CONFIGURACIÓN DEL ENTORNO Y CARGA DE DATOS
# Configuramos Pandas para que los textos largos y las URLs se lean completos
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

# Cargamos el archivo maestro refinado (más de 6,000 registros limpios)
archivo_maestro = 'catalogo_tecnologia_maestro_5500_refinado.csv'
df_maestro = pd.read_csv(archivo_maestro)

# Definimos estrictamente las 8 columnas clave para una presentación ejecutiva
columnas_relevantes = [
    'marca',
    'nombre',
    'modelo',
    'color',
    'precio',
    'moneda',
    'categoria',
    'url'
]


# 2. CREACIÓN DEL MENÚ DE OPCIONES INTERACTIVO (@interact)
# Usamos el decorador @interact de ipywidgets. Esto genera automáticamente un
# panel gráfico superior con campos de texto, menús desplegables y controles deslizantes.

@interact(
    marca=widgets.Dropdown(
        options=['Todas', 'Apple', 'Samsung'],
        value='Todas',
        description='Marca:'
    ),
    buscar_palabra=widgets.Text(
        value='',
        placeholder='Escribe ej. Watch, Pro, Galaxy...',
        description='Búsqueda:'
    ),
    limite_resultados=widgets.IntSlider(
        min=5, max=50, step=5, value=15,
        description='Mostrar:'
    )
)
def buscador_interactivo_catalogo(marca, buscar_palabra, limite_resultados):
    """
    Función controladora del menú. Filtra dinámicamente el DataFrame maestro
    en función de la marca seleccionada y la palabra clave ingresada.
    """
    # Copia del DataFrame para mantener la integridad de los datos originales
    df_filtrado = df_maestro.copy()

    # 2.1. Filtro por Marca seleccionada en el menú desplegable
    if marca != 'Todas':
        df_filtrado = df_filtrado[df_filtrado['marca'] == marca]

    # 2.2. Filtro por Término de Búsqueda (ignorando mayúsculas/minúsculas y acentos)
    termino = buscar_palabra.strip().lower()
    if termino:
        df_filtrado = df_filtrado[
            df_filtrado['nombre'].str.lower().str.contains(termino, na=False) |
            df_filtrado['descripcion'].str.lower().str.contains(termino, na=False) |
            df_filtrado['modelo'].str.lower().str.contains(termino, na=False)
        ]

    # 2.3. Despliegue de Estadísticas del Filtro en Pantalla
    total_encontrados = len(df_filtrado)

    display(HTML(f"""
        <div style="background-color: #f4f6f9; padding: 10px; border-radius: 5px; margin-bottom: 10px; font-family: Arial;">
            <b> Panel de Resultados:</b> Se encontraron <b>{total_encontrados}</b> coincidencias
            para la marca <i>'{marca}'</i> con el término <i>'{termino if termino else "Ninguno"}'</i>.
        </div>
    """))

    # 2.4. Mostrar la tabla resultante limitada al número seleccionado en el slider
    if total_encontrados > 0:
        display(df_filtrado[columnas_relevantes].head(limite_resultados))
    else:
        print(" No se encontraron productos que coincidan con los criterios especificados.")

interactive(children=(Dropdown(description='Marca:', options=('Todas', 'Apple', 'Samsung'), value='Todas'), Te…

**Justificacion:**\
Para dar cumplimiento al rubro de Resultados de búsqueda dentro del proyecto, se desarrolló un motor de consulta automatizado basado en Pandas e interfaces interactivas de ipywidgets. El propósito de este módulo es permitir al usuario final (o al evaluador) realizar búsquedas granulares y dinámicas sobre el catálogo de Apple y Samsung ($N = 6,035$ registros)